# Causal Reasoning in NNs

## Causal Reasoning in Neural Networks

Standard ML learns correlations: P(Y|X). Causal ML asks: what is the effect of *intervening* on X, i.e., P(Y|do(X=x))? Pearl's do-calculus formalizes this distinction.

**Structural Causal Models (SCMs)**: Variables + structural equations (assignments with noise) + a directed acyclic graph.

**Interventions (do-operator)**: Remove all incoming edges to a variable and set it to a fixed value.

**Counterfactuals**: Given that Y=y occurred with X=x, what would Y be if we had set X=x'?

**Neural causal models**: Learn structural equations with neural networks from observational data, then perform causal inference.

In [ ]:
import numpy as np
from typing import Dict, List

np.random.seed(42)

# Structural Causal Model: a simple medical scenario
# Drug -> Recovery; Age -> Drug, Recovery; Severity -> Drug, Recovery

class StructuralCausalModel:
    def __init__(self):
        self.equations = {}
        self.noise_fns = {}
    
    def add_equation(self, variable, parents, fn):
        self.equations[variable] = {'parents': parents, 'fn': fn}
    
    def sample(self, n=1000, interventions=None):
        interventions = interventions or {}
        data = {}
        # Topological order: age, severity, drug, recovery
        for var in ['age', 'severity', 'drug', 'recovery']:
            if var in interventions:
                data[var] = np.full(n, interventions[var], dtype=float)
            else:
                parents = self.equations[var]['parents']
                parent_vals = {p: data[p] for p in parents}
                data[var] = self.equations[var]['fn'](parent_vals, n)
        return data

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Define the SCM
scm = StructuralCausalModel()
scm.add_equation('age', [], lambda p, n: np.random.normal(50, 15, n).clip(18, 90))
scm.add_equation('severity', [], lambda p, n: np.random.beta(2, 5, n))  # 0-1
scm.add_equation('drug', ['age', 'severity'], 
    lambda p, n: (sigmoid(0.02*(p['age']-60) + 2*p['severity']) > 
                  np.random.uniform(0, 1, n)).astype(float))
scm.add_equation('recovery', ['age', 'severity', 'drug'],
    lambda p, n: (sigmoid(-0.01*p['age'] - 2*p['severity'] + 1.5*p['drug'] + 
                          np.random.normal(0, 0.3, n)) > 0.5).astype(float))

# Observational vs interventional distribution
obs = scm.sample(10000)
int_drug1 = scm.sample(10000, interventions={'drug': 1})  # everyone gets drug
int_drug0 = scm.sample(10000, interventions={'drug': 0})  # no one gets drug

print('Causal Reasoning: Drug Effect on Recovery')
print('=' * 50)
print(f'Observational P(recovery | drug=1): '
      f'{obs["recovery"][obs["drug"]==1].mean():.3f}')
print(f'Observational P(recovery | drug=0): '
      f'{obs["recovery"][obs["drug"]==0].mean():.3f}')
print(f'Causal P(recovery | do(drug=1)):    {int_drug1["recovery"].mean():.3f}')
print(f'Causal P(recovery | do(drug=0)):    {int_drug0["recovery"].mean():.3f}')
print(f'ATE = E[Y(1)] - E[Y(0)] = '
      f'{int_drug1["recovery"].mean() - int_drug0["recovery"].mean():.3f}')
print('\nNote: Observational correlation differs from causal effect')
print('due to age/severity confounders (older/sicker patients get drug more)')
